In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

import transformers
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from dataset import CounterfactualAudioDataset
from models import AudioTextCounterfactualModel

transformers.logging.set_verbosity_error()

In [ ]:
class CounterfactualLoss(nn.Module):
    """
    Composite Loss function combining Factual Consistency and Angle Loss
    """
    def __init__(self, margin=0.1, w1=1.0, w2=100.0):
        super().__init__()
        self.margin = margin
        self.w1 = w1  # Angle Loss Weight
        self.w2 = w2  # Factual Consistency Weight
        self.mse = nn.MSELoss()

    def forward(self, audio_embeds, factual_embeds, counterfactual_embeds):
        # Equation 5: Factual Consistency Loss (Mean Squared Error)
        # Drives the audio embedding towards the factual caption
        l_factual = self.mse(audio_embeds, factual_embeds)

        # Equation 3 & 4: Angle Loss (Triplet Margin Cosine)
        # Cosine similarity is the dot product of L2 normalized vectors
        cos_factual = torch.sum(audio_embeds * factual_embeds, dim=-1)
        cos_counterfactual = torch.sum(audio_embeds * counterfactual_embeds, dim=-1)

        # Punishes the model if the counterfactual similarity is greater than the factual similarity
        # L_angle = max(0, cos(a, cf) - cos(a, f) + margin)
        l_angle = torch.mean(torch.clamp(cos_counterfactual - cos_factual + self.margin, min=0.0))

        # Equation 6: Total Composite Loss
        total_loss = (self.w1 * l_angle) + (self.w2 * l_factual)

        return total_loss, l_angle, l_factual

In [ ]:
def evaluate_retrieval(model, dataloader, device):
    """
    Evaluates the model on the Language-Based Audio Retrieval task (Table 2).
    """
    model.eval()
    all_audio_embeds = []
    all_text_embeds = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            audio, captions, _ = batch
            audio = audio.to(device)

            audio_embeds = model.encode_audio(audio)
            text_embeds = model.encode_text(captions, device)
            
            all_audio_embeds.append(audio_embeds.cpu())
            all_text_embeds.append(text_embeds.cpu())

    all_audio_embeds = torch.cat(all_audio_embeds, dim=0)
    all_text_embeds = torch.cat(all_text_embeds, dim=0)

    # Compute Cosine Similarity Matrix (N_text_queries, N_audio_database)
    # Because vectors are normalized, matrix multiplication yields cosine similarity
    sim_matrix = torch.matmul(all_text_embeds, all_audio_embeds.T)

    N = sim_matrix.shape[0]
    top1_correct = 0
    top10_correct = 0

    # Calculate Top-k Recall
    for i in range(N):
        ranked_indices = torch.argsort(sim_matrix[i], descending=True)

        # The correct audio for text query `i` is at index `i`
        if ranked_indices[0] == i:
            top1_correct += 1

        if i in ranked_indices[:10]:
            top10_correct += 1

    top1_acc = top1_correct / N
    top10_acc = top10_correct / N

    return top1_acc, top10_acc

In [ ]:
device_type = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(device_type)
print(f"Using device: {device}")

# Hyperparameters from paper (Table 4 optimal setup)
bs = 32
epochs = 15
lr = 1e-4
w1 = 1.0     # Angle Loss weight
w2 = 100.0   # Factual Consistency weight

resume_checkpoint = "models/checkpoint.pth"

train_dataset = CounterfactualAudioDataset("data/metadata.csv")
test_dataset = CounterfactualAudioDataset("data/clotho_eval_metadata.csv") 

train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True, num_workers=20, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=bs, shuffle=False, num_workers=20)

model = AudioTextCounterfactualModel().to(device)
criterion = CounterfactualLoss(margin=0.1, w1=w1, w2=w2)

optimizer = torch.optim.AdamW(model.audio_encoder.parameters(), lr=lr)
scaler = torch.GradScaler(device_type)

start_epoch = 0
if resume_checkpoint and os.path.exists(resume_checkpoint):
    print(f"Loading checkpoint '{resume_checkpoint}'...")
    checkpoint = torch.load(resume_checkpoint, map_location=device)
    model.audio_encoder.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if 'scaler_state_dict' in checkpoint:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    print(f"Resumed training from epoch {start_epoch + 1}")
elif resume_checkpoint:
    print(f"Checkpoint '{resume_checkpoint}' not found. Starting from scratch.")

In [ ]:
for epoch in range(start_epoch, epochs):
    model.train()
    total_loss_epoch = 0

    progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{epochs}")
    for batch_idx, batch in progress_bar:
        audio, factual_captions, counterfactual_captions = batch
        audio = audio.to(device)

        optimizer.zero_grad()

        audio_embeds = model.encode_audio(audio)
        factual_embeds = model.encode_text(factual_captions, device)
        counterfactual_embeds = model.encode_text(counterfactual_captions, device)

        loss, l_angle, l_factual = criterion(audio_embeds, factual_embeds, counterfactual_embeds)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.audio_encoder.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss_epoch += loss.item()
        progress_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

        if batch_idx == len(train_loader) - 1:
            progress_bar.set_postfix({"Loss": f"{loss.item():.4f}", "Avg Loss": f"{total_loss_epoch/len(train_loader):.4f}"})

    if (epoch + 1) % 5 == 0:  # Save checkpoint every epoch
        checkpoint_path = f"models/checkpoint_epoch_{epoch+1}.pth"
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.audio_encoder.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, checkpoint_path)
        print(f"Checkpoint saved at '{checkpoint_path}'")


torch.save(model.audio_encoder.state_dict(), "models/counterfactual_audio_encoder.pth")
print("Training Finished. Model weights saved.")

In [ ]:
top1_acc, top10_acc = evaluate_retrieval(model, test_loader, device)
print(f"Top-1 Accuracy: {top1_acc:.4f}")
print(f"Top-10 Accuracy: {top10_acc:.4f}")

### Mixed Precision (NaN problems)

In [ ]:
for epoch in range(start_epoch, epochs):
    model.train()
    total_loss_epoch = 0

    progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{epochs}")
    for batch_idx, batch in progress_bar:
        audio, factual_captions, counterfactual_captions = batch
        audio = audio.to(device)

        optimizer.zero_grad()

        with torch.autocast(device_type):
            audio_embeds = model.encode_audio(audio)
            factual_embeds = model.encode_text(factual_captions, device)
            counterfactual_embeds = model.encode_text(counterfactual_captions, device)

        loss, l_angle, l_factual = criterion(audio_embeds, factual_embeds, counterfactual_embeds)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.audio_encoder.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss_epoch += loss.item()
        progress_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

        if batch_idx == len(train_loader) - 1:
            progress_bar.set_postfix({"Loss": f"{loss.item():.4f}", "Avg Loss": f"{total_loss_epoch/len(train_loader):.4f}"})

    if (epoch + 1) % 5 == 0:  # Save checkpoint every epoch
        checkpoint_path = f"models/checkpoint_epoch_{epoch+1}.pth"
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.audio_encoder.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
        }, checkpoint_path)
        print(f"Checkpoint saved at '{checkpoint_path}'")

torch.save(model.audio_encoder.state_dict(), "models/counterfactual_audio_encoder.pth")
print("Training Finished. Model weights saved.")